# SOTA PDF parsing + cleaning (RAG)

**Google Colab + Google Drive only.** No zip uploads in the notebook.

### One-time setup on your computer

1. In [Google Drive](https://drive.google.com), create a folder, for example **`RAG`**.
2. Inside it, create a folder named **`docs`** and put all your `.pdf` files there.

Your Drive layout should look like:

```
My Drive/
  RAG/
    docs/          ← PDFs go here
    parsed_out/    ← created automatically by this notebook
```

3. Open this notebook in Colab and run all cells. When asked, **allow access to Google Drive**.

### What the notebook does

- **Reads** PDFs from `My Drive/RAG/docs`
- **Writes** Markdown + `manifest.jsonl` to `My Drive/RAG/parsed_out`
- Uses **Docling** (layout/tables) and **PyMuPDF4LLM** (fast RAG-friendly Markdown), then **cleans** the text

If your folder name is not `RAG`, change `PROJECT_FOLDER` in the setup cell below.

## 1) Install dependencies

Run once per new Colab runtime (~2–5 min). Docling pulls PyTorch and doc models; use **GPU runtime** for faster OCR/VLM workloads if you enable them later.

In [ ]:
%%capture
# Core SOTA-oriented stack (versions float; pin in production if you need reproducibility)
!pip install -q docling pymupdf4llm pymupdf ftfy tqdm orjson

import os
import sys
os.environ["DOCLING_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
print("Python:", sys.version)

## 2) Mount Google Drive and set paths

Run this cell once per Colab session. Approve the Drive permission prompt in the browser.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# ------------- EDIT ONLY IF YOUR DRIVE FOLDER NAME DIFFERS -------------
PROJECT_FOLDER = "RAG"   # folder under "My Drive"
DOCS_SUBFOLDER = "docs"           # PDFs live here
OUTPUT_SUBFOLDER = "parsed_out"   # results written here

USE_DOCLING = False                # layout-aware (recommended for reports)
USE_PYMUPDF4LLM = True            # fast Markdown path for RAG
DOCLING_OCR = False               # True only for scanned/image PDFs (slower)
# ----------------------------------------------------------------------

MYDRIVE = Path("/content/drive/MyDrive")
PROJECT_DIR = MYDRIVE / PROJECT_FOLDER
DOCS_DIR = PROJECT_DIR / DOCS_SUBFOLDER
OUTPUT_DIR = PROJECT_DIR / OUTPUT_SUBFOLDER

for sub in ("markdown_docling", "markdown_pymupdf4llm", "cleaned"):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Input  (PDFs):", DOCS_DIR, "-> exists:", DOCS_DIR.exists())
print("Output (parsed):", OUTPUT_DIR)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Create this folder in Google Drive and upload your PDFs:\n"
        f"  {PROJECT_DIR}\n"
        f"  +-- {DOCS_SUBFOLDER}/   (put .pdf files here)"
    )

if not DOCS_DIR.exists():
    raise FileNotFoundError(
        f"Missing docs folder on Drive. Create:\n  {DOCS_DIR}\n"
        f"and upload your PDF files into it."
    )

pdfs = sorted(DOCS_DIR.rglob("*.pdf"))
print(f"PDFs found on Drive: {len(pdfs)}")
if not pdfs:
    raise FileNotFoundError(f"No .pdf files under {DOCS_DIR}")

## 3) Cleaning utilities

These fixes target typical PDF extraction issues: broken line hyphenation, ligatures, odd spaces, and runaway blank lines.

In [ ]:
import re
import unicodedata
import ftfy

def normalize_unicode(text: str) -> str:
    text = ftfy.fix_text(text)
    text = unicodedata.normalize("NFKC", text)
    return text

def fix_hyphenation_linebreaks(text: str) -> str:
    # "exam-\nple" -> "example"
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)
    return text

def collapse_blank_lines(text: str, max_blank: int = 2) -> str:
    text = re.sub(r"\n{3,}", "\n" * max_blank, text)
    return text

def strip_per_line_whitespace(text: str) -> str:
    lines = [ln.rstrip() for ln in text.splitlines()]
    return "\n".join(lines).strip() + "\n"

def remove_nulls(text: str) -> str:
    return text.replace("\x00", "")

def clean_markdown(text: str) -> str:
    text = remove_nulls(text)
    text = normalize_unicode(text)
    text = fix_hyphenation_linebreaks(text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = collapse_blank_lines(text, max_blank=2)
    text = strip_per_line_whitespace(text)
    return text

print("clean_markdown OK")

## 4) Parse all PDFs from Drive

Writes to **`My Drive/RAG/parsed_out/`**:

- `markdown_docling/*.md` — Docling export (structure-preserving).
- `markdown_pymupdf4llm/*.md` — PyMuPDF4LLM export (RAG-friendly).
- `cleaned/*.md` — cleaned copies (both sources if enabled).
- `manifest.jsonl` — one JSON object per file with paths, errors, and lengths.

In [ ]:
import shutil
import json
import re
import traceback
from datetime import datetime, timezone
from tqdm.auto import tqdm

import orjson


def safe_stem(p: Path) -> str:
    # filesystem-safe stem for outputs
    stem = p.stem
    stem = re.sub(r"[^\w\-]+", "_", stem, flags=re.UNICODE)
    stem = re.sub(r"_+", "_", stem).strip("_")
    return stem[:180] if stem else "document"


def write_jsonl_row(path: Path, row: dict) -> None:
    line = orjson.dumps(row, option=orjson.OPT_APPEND_NEWLINE)
    with open(path, "ab") as f:
        f.write(line)


manifest_path = OUTPUT_DIR / "manifest.jsonl"
if manifest_path.exists():
    manifest_path.unlink()

pdf_paths = sorted(DOCS_DIR.rglob("*.pdf"))
print(f"Parsing {len(pdf_paths)} PDFs from Drive...")

# Lazy imports so install cell can be run standalone
if USE_DOCLING:
    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
    from docling.datamodel.base_models import InputFormat
    try:
        from docling.datamodel.pipeline_options import EasyOcrOptions
    except Exception:  # pragma: no cover
        EasyOcrOptions = None  # type: ignore

if USE_PYMUPDF4LLM:
    import pymupdf4llm
    # Disable heavy table detection/layout parsing which hangs on complex vectors/charts
    pymupdf4llm.use_layout(False)


def build_docling_converter():
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = bool(DOCLING_OCR)
    pipeline_options.table_structure_options.mode = TableFormerMode.FAST
    pipeline_options.generate_picture_images = False
    pipeline_options.generate_table_images = False
    if DOCLING_OCR and EasyOcrOptions is not None:
        # EasyOCR backend inside Docling; first run may download models
        pipeline_options.ocr_options = EasyOcrOptions(lang=["en"])
    elif DOCLING_OCR:
        print("WARN: EasyOcrOptions unavailable; Docling will use default OCR/auto settings.")
    format_options = {
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
    return DocumentConverter(format_options=format_options)


converter = build_docling_converter() if USE_DOCLING else None

for pdf_path in tqdm(pdf_paths, desc="PDFs"):
    stem = safe_stem(pdf_path)
    row = {
        "source_pdf": str(pdf_path),
        "stem": stem,
        "utc_iso": datetime.now(timezone.utc).isoformat(),
        "docling_md": None,
        "pymupdf4llm_md": None,
        "cleaned_docling_md": None,
        "cleaned_pymupdf_md": None,
        "errors": [],
    }

    # 1) Copy PDF to local SSD to bypass Google Drive network lag
    local_pdf = Path("/content/temp.pdf")
    try:
        shutil.copy(str(pdf_path), str(local_pdf))
    except Exception as e:
        row["errors"].append({"stage": "local_copy", "trace": str(e)})
        write_jsonl_row(manifest_path, row)
        continue

    # 2) Parse the fast local copy
    if USE_DOCLING and converter is not None:
        try:
            result = converter.convert(str(local_pdf))
            md = result.document.export_to_markdown()
            out = OUTPUT_DIR / "markdown_docling" / f"{stem}.md"
            out.write_text(md, encoding="utf-8")
            row["docling_md"] = str(out)
            cpath = OUTPUT_DIR / "cleaned" / f"{stem}__docling.md"
            cpath.write_text(clean_markdown(md), encoding="utf-8")
            row["cleaned_docling_md"] = str(cpath)
        except Exception:
            row["errors"].append({"stage": "docling", "trace": traceback.format_exc()})

    if USE_PYMUPDF4LLM:
        try:
            md2 = pymupdf4llm.to_markdown(str(local_pdf))
            out2 = OUTPUT_DIR / "markdown_pymupdf4llm" / f"{stem}.md"
            out2.write_text(md2, encoding="utf-8")
            row["pymupdf4llm_md"] = str(out2)
            cpath2 = OUTPUT_DIR / "cleaned" / f"{stem}__pymupdf4llm.md"
            cpath2.write_text(clean_markdown(md2), encoding="utf-8")
            row["cleaned_pymupdf_md"] = str(cpath2)
        except Exception:
            row["errors"].append({"stage": "pymupdf4llm", "trace": traceback.format_exc()})

    # 3) Clean up local PDF
    if local_pdf.exists():
        local_pdf.unlink()

    write_jsonl_row(manifest_path, row)

print("Done. Manifest:", manifest_path)
print("Sample rows (first 2):")
with open(manifest_path, "rb") as f:
    for _ in range(2):
        line = f.readline()
        if not line:
            break
        print(json.dumps(json.loads(line), indent=2)[:1200])

## 5) Quick QA — preview one cleaned file

Change `stem` to match a `*.md` in `parsed_out/cleaned/` (see filename without extension).

In [ ]:
from IPython.display import Markdown, display

cleaned_dir = OUTPUT_DIR / "cleaned"
candidates = sorted(cleaned_dir.glob("*.md"))
print("n_cleaned_files:", len(candidates))
if candidates:
    sample = candidates[0]
    print("Preview:", sample)
    text = sample.read_text(encoding="utf-8", errors="replace")
    display(Markdown(text[:6000] + ("\n\n…(truncated)…" if len(text) > 6000 else "")))

## 6) Results on Google Drive

All outputs are already saved under `parsed_out` on Drive (no extra download step).

In [ ]:
from pathlib import Path

print("Done. Open this folder in Google Drive:")
print(" ", OUTPUT_DIR)
print()
print("Contents:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        rel = p.relative_to(OUTPUT_DIR)
        print(f"  {rel}  ({p.stat().st_size // 1024} KB)")

---

### Notes for your Malaysia statistical PDFs

- **Digital PDFs:** leave `DOCLING_OCR = False` for speed.
- **Scanned reports:** set `DOCLING_OCR = True` (install time and runtime increase; GPU helps).
- **Very large files:** consider processing in batches or raising Colab disk/RAM; Docling is heavier than PyMuPDF4LLM alone.
- **RAG chunking:** use cleaned Docling MD for structure-heavy tables; compare with PyMuPDF4LLM MD if chunks look noisy.

### Optional cloud APIs (not in this notebook)

If you later want vendor-hosted “SOTA” (LlamaParse, Reducto, etc.), swap the parse step but keep the cleaning + manifest pattern.